In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import PromptTemplate
from langchain import hub
from langchain_core.tools import tool
from datetime import datetime

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# https://smith.langchain.com/hub/hwchase17/react
# 특정 사용자 hwchase17
prompt_template = hub.pull("hwchase17/react")
print(prompt_template.template)

In [ ]:
# 도구 호출 기능이란 단순 텍스트 생성이 아니라
# 사용자의 요청을 분석해서 1. 외부 도구 사용을 판단 2. 도구 호출 지시를 생성하는 기능
# 도구 호출 기능이 없는 언어 모델의 출력: "서버가 다운된 것 같아. IT 부서에 전화해야 할 것 같아. 전화할 번호는 123456이야."
# 도구 호출 기능이 있는 언어 모델의 출력: "call_phone(number='123456')"

In [ ]:

# @tool 데코레이터의 역할
# 해당 함수는 에이전트가 호출할 수 있는 도구임을 알리는 역할
# 함수가 어떤 기능을 수행하는지, 어떤 매개변수를 필요로 하는지, 그리고 어떤 결과를 반환하는지에 대한 정보를 LLM에게 알려줌
@tool
def get_current_time() -> str:
    """Returns the current time in the format 'YYYY-MM-DD HH:MM:SS'."""
    # 함수의 docstring은 함수의 첫 번째 문장에 세 개의 따옴표(""" 또는 ''')로 감싸서 작성
    # 함수의 docstring은 언어 모델에게 도구의 역할을 설명하는 내용
    # docstring은 언어 모델이 해당 함수를 사용할지 말지 결정하는 기준
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

llm = ChatOpenAI(model="gpt-4o", temperature=0) # 도구 호출 기능
tools = [get_current_time]

prompt_template = hub.pull("hwchase17/react")

# 에이전트 생성
# 에이전트는 tools을 통하여 도구에 대한 정보를 언어 모델에 전달
# create_react_agent vs. create_tool_calling_agent
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt_template
    # 
)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
# agent_executor = AgentExecutor(agent=agent, tools=tools)

# 에이전트 실행
# 'verbose=True' 설정으로 언어 모델의 추론 과정 출력
response = agent_executor.invoke({"input": "지금 몇 시야?"})

print(f"\n최종 답변: {response['output']}")

In [ ]:
# 위 결과는 세 번의 출력을 한 것
# 1. Action과 Action Input 출력
# 2. Thought와 Final Answer를 출력

# 1. Action과 Action Input 출력
# Action: get_current_time
# Action Input: None

# 2. Thought와 Final Answer를 출력
# I now know the final answer
# Final Answer: 지금은 2025년 9월 12일 14시 12분 56초입니다.

# 중간의 색깔이 다른 출력 2025-09-12 14:12:56은 에이전트가 출력한 것

# ReAct 방식에서 언어 모델의 출력은 (Action과 Action Input 출력)을 반복하다가 Thought와 Final Answer를 출력
# Thought는 생략되기도 함
# Observation(중간의 색깔이 다른 출력)은 에이전트가 출력

In [ ]:
# create_react_agent의 한계

from langchain_google_genai import ChatGoogleGenerativeAI

@tool
def add_numbers(a: int, b: int) -> int:
    # """두 개의 정수를 더합니다. Add two integers."""
    """두 개의 정수를 더합니다. 예시: `add_numbers(a=2, b=3)` -> 5"""
    return a + b

@tool
def multiply_numbers(a: int, b: int) -> int:
    # """두 개의 정수를 곱합니다. Multiply two integers."""
    # """두 개의 정수를 곱합니다. 예시: `multiply_numbers(a=2, b=3)` -> 6"""
    """a와 b는 반드시 정수로 변환해야 합니다"""
    return a * b

llm = ChatOpenAI(model="gpt-4o", temperature=0)
# llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
tools = [add_numbers, multiply_numbers] # 두 개의 사용자 정의 도구를 목록에 추가

prompt_template = hub.pull("hwchase17/react")

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt_template
)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

response = agent_executor.invoke({"input": "5 더하기 3을 한 다음에 그 결과에 2를 곱해줘."})

print(f"\n최종 답변: {response['output']}")

In [ ]:
# ReAct 에이전트의 어려운 점은 언어 모델이 도구가 요구하는 특정 형식의 입력(정형화된 데이터)을 일관성 있게 생성하는 것이 어렵다는 점

In [ ]:
# Agent와 AgentExecutor의 역할 분담

In [ ]:
# AgentExecutor

# AgentExecutor는 Agent가 넘겨준 파싱 결과물이

# AgentAction이면 AgentExecutor가 해당 Tool을 실행하고 (AgentAction, observation)을 
# intermediate_steps에 추가한 뒤, 다시 Agent를 호출해서 루프

# AgentFinish이면 더 이상 도구 호출 필요 없다고 판단, 루프를 종료하고 최종 답변을 반환

In [ ]:
# 언어 모델(LLM)의 raw 출력을 제일 처음 접하는 주체는 Agent
# Agent는 언어모델의 raw 문자열 출력을 파싱해서 AgentAction 또는 AgentFinish로 변환한 뒤 AgentExecutor에게 전달
# 즉, Agent는 도구 실행을 계획하고 지시하는 역할을 하고 실제 도구 호출은 AgentExecutor가 수행

# AgentExecutor는 Agent로부터 받은 결과가 
# AgentAction이면 지정 툴을 실행하고 그 결과를 
# (AgentAction, str) 형태로 intermediate_steps에 저장, str은 그 결과에 해당하는 observation
# AgentExecutor는 intermediate_steps를 Agent에 전달

# AgentExecutor는 Agent로부터 받은 결과가 AgentFinish이면 최종 답변으로 간주하고 실행을 종료

In [ ]:
# AgentExecutor가 내부적으로 기록하는 로그가 intermediate_steps
# intermediate_steps는 지금까지 진행된 모든 도구 호출 기록이 기록되어 있음

# Agent는 intermediate_steps를 읽기만 하고 프롬프트에 활용

# (AgentAction, str)의 리스트로서
# [(AgentAction, str), (AgentAction, str), (AgentAction, str), (AgentAction, str),...]의 형태

# (AgentAction, str)에서
# AgentAction는 에이전트가 어떤 도구(tool)를 어떻게 호출했는지를 나타냄
# -tool (사용한 도구 이름, str)
# -tool_input (도구에 넣은 입력값, str/dict 등)
# -log (LLM이 출력한 원본 텍스트 일부)

# str은 도구 실행의 출력값 (실제 결과)

intermediate_steps = [
    (
        AgentAction(
            tool="Calculator",
            tool_input="2+2",
            log="I will use Calculator to solve 2+2"
        ),
        "4"
    ),
    (
        AgentAction(
            tool="Wikipedia",
            tool_input="Python programming",
            log="Searching for Python programming"
        ),
        "Python is a high-level programming language..."
    )
]


In [ ]:
# AgentFinish는 에이전트 실행이 완료되었을 때 반환되는 최종 결과와 로그를 담고 있는 논리적인 구조

{
  "return_values": {
    "output": "현재 서울의 날씨는 맑고 25도입니다."
  },
  "log": "I now know the final answer after looking up the weather and am ready to respond."
}

In [ ]:
# AgentExecutor가 지금까지 쌓아온 intermediate_steps를 Agent에게 전달
# Agent는 intermediate_steps를 받아서 scratchpad로 변환

In [ ]:
# intermediate_steps를 언어 모델이 이해할 수 있도록 바꾼 형태가 scratchpad
# 여기서 이해한다는 의미는 프롬프트의 일부가 되게 한다는 것을 의미
# 에이전트의 도구 호출 기록을 언어 모델에 전달하는 이유는 언어 모델이 다음 단계를 결정하는데
# 기억으로서의 역할을 하는 것

[
    (
        AgentAction(tool="Calculator", tool_input="2+2", log="I will calculate 2+2"),
        "4"
    )
]

# 그런데,
# intermediate_steps → scratchpad 변환은 크게 두 가지 방법이 있음
# 1. ReAct 스타일
# 2. Tool Calling 스타일


In [ ]:
# 1. ReAct 스타일에서는 intermediate_steps의 한 튜플을 Thought → Action → Action Input → Observation 포맷으로 변환

# AgentAction.log → Thought
# AgentAction.tool → Action
# AgentAction.tool_input → Action Input
# str (tuple의 두 번째 값, 도구 실행 결과) → Observation

# 아래와 같이 변환되어서 프롬프트 내부의 {scratch_pad}에 채워짐
[
    (
        AgentAction(tool="Calculator", tool_input="2+2", log="I will calculate 2+2"),
        "4"
    )
]
# Thought: I will calculate 2+2
# Action: Calculator
# Action Input: 2+2
# Observation: 4

In [ ]:
# 2. Tool Calling 스타일의 경우 원래의 intermediate_steps의 (AgentAction, str)를
# 규칙
# AgentAction는 AIMessage로 바뀜
# AIMessage.additional_kwargs["tool_calls"] 내에 기록
# str(결과) → FunctionMessage (Observation)

[
    (
        AgentAction(tool="Calculator", tool_input="2+2", log="I will calculate 2+2"),
        "4"
    )
]


[
    AIMessage(
        content="",
        additional_kwargs={
            "tool_calls": [
                {
                    "id": "call_1",
                    "function": {
                        "name": "Calculator",
                        "arguments": '{"expression": "2+2"}'
                    }
                }
            ]
        }
    ),
    FunctionMessage(
        name="Calculator",
        content="4"
    )
]


In [ ]:
# 언어 모델의 출력 - 도구 호출 기능이 없는 언어 모델

# 도구 호출 기능이 없는 언어 모델은 텍스트만 만들 수 있고 JSON 같은 구조적인 값은 만들지 못함
# 언어 모델의 출력은 결국 프롬프트를 통해서 그렇게 유도한 결과

# 1. 도구 호출이 필요 없는 경우에는
# Thought: 이 질문은 바로 답할 수 있다.
# Final Answer: 파리의 에펠탑은 1889년에 건설되었다.
# 이 경우, Agent는 AgentFinish로 변경하며 Agent는 Thought 부분을 파싱 결과에 추가하지 않음 (log에는 남음)

# 2. 도구 호출을 해야 하는 경우에는
# Thought: 최신 정보가 필요하다.
# Action: Search
# Action Input: "에펠탑 최신 방문객 수"
# 이 경우, Agent는 AgentAction으로 변경하며 Agent는 Thought 부분을 파싱 결과에 추가하지 않음 (log에는 남음)

In [ ]:
################ 강의에서 사용한 코드 (에러가 발생하지 않음을 확인하고 사용했다가 나중에 다시 에러가 발생)
################ 강의에서 사용한 코드 (에러가 발생하지 않음을 확인하고 사용했다가 나중에 다시 에러가 발생)
################ 강의에서 사용한 코드 (에러가 발생하지 않음을 확인하고 사용했다가 나중에 다시 에러가 발생)
################ 강의에서 사용한 코드 (에러가 발생하지 않음을 확인하고 사용했다가 나중에 다시 에러가 발생)
################ 강의에서 사용한 코드 (에러가 발생하지 않음을 확인하고 사용했다가 나중에 다시 에러가 발생)

import re
import ast
from langchain.agents.agent import AgentOutputParser
from langchain_core.agents import AgentAction, AgentFinish
from langchain_core.exceptions import OutputParserException
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import AgentExecutor
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain import hub
from langchain.agents.format_scratchpad import format_log_to_str

# AgentOutputParser는 추상 클래스로서 parse() 메서드를 구현해야 함
# parse() 메서드를 구현함으로써 내가 정의한 방식대로 언어 모델의 출력을 해석하는 Agent를 만들 수 있음
# parse는 맘대로 구현해도 되지만 parse의 결과물은 AgentAction/AgentFinish 객체라는 조건은 지켜야 Agent 역할을 할 수 있음

class CustomReactOutputParser(AgentOutputParser):
    def parse(self, text: str) -> AgentAction | AgentFinish:
        if "Final Answer:" in text:
            final_answer = text.split("Final Answer:")[-1].strip()
            return AgentFinish(
                return_values={"output": final_answer},
                log=text,
            )
        action_re = re.compile(r"Action: (.*?)[\n\r]*Action Input: (.*)", re.DOTALL)
        match = action_re.search(text)
        if not match:
            raise OutputParserException(f"Could not parse LLM output: `{text}`")

        action = match.group(1).strip()
        action_input_str = match.group(2).strip()

        # 수정된 부분: Action Input을 파싱하는 로직을 통합
        tool_input = {}
        try:
            # 첫 번째 시도: ast.literal_eval로 딕셔너리 문자열을 평가
            evaluated_input = ast.literal_eval(action_input_str)
            if isinstance(evaluated_input, dict):
                tool_input = evaluated_input
            else:
                # ast.literal_eval이 실패하거나 딕셔너리가 아닌 경우
                raise ValueError
        except (SyntaxError, ValueError, TypeError):
            # 두 번째 시도: '키=값' 패턴을 정규 표현식으로 찾아서 딕셔너리로 변환
            key_value_pairs = re.findall(r"(\w+)\s*=\s*([0-9.]+)", action_input_str)
            if not key_value_pairs:
                raise OutputParserException(f"Could not parse action input: `{action_input_str}`")

            try:
                # key_value_pairs가 비어 있지 않으면, 딕셔너리 생성
                tool_input = {key: int(value) for key, value in key_value_pairs}
            except ValueError:
                raise OutputParserException(f"Could not convert action input values to integers: `{action_input_str}`")

        return AgentAction(tool=action, tool_input=tool_input, log=text)


In [ ]:
################ 강의에서 사용한 코드 (에러가 발생하지 않음을 확인하고 사용했다가 나중에 다시 에러가 발생)
################ 강의에서 사용한 코드 (에러가 발생하지 않음을 확인하고 사용했다가 나중에 다시 에러가 발생)
################ 강의에서 사용한 코드 (에러가 발생하지 않음을 확인하고 사용했다가 나중에 다시 에러가 발생)
################ 강의에서 사용한 코드 (에러가 발생하지 않음을 확인하고 사용했다가 나중에 다시 에러가 발생)
################ 강의에서 사용한 코드 (에러가 발생하지 않음을 확인하고 사용했다가 나중에 다시 에러가 발생)

@tool
def add_numbers(a: int, b: int) -> int:
    """두 개의 정수를 더합니다. 예시: `add_numbers(a=2, b=3)` -> 5"""
    return a + b

@tool
def multiply_numbers(a: int, b: int) -> int:
    """두 개의 정수를 곱합니다. 예시: `multiply_numbers(a=2, b=3)` -> 6"""
    return a * b

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
tools = [add_numbers, multiply_numbers]
tool_names = ", ".join([t.name for t in tools])
tools_info = "\n".join([f"{t.name}: {t.description}" for t in tools])
# t.name: 함수 이름
# t.description: DocString

prompt_template = hub.pull("hwchase17/react")

from langchain.agents.format_scratchpad import format_log_to_str

# 전달되는 값은 { "input": ..., "intermediate_steps": [] }의 형태
agent_chain = RunnablePassthrough.assign(
    agent_scratchpad=lambda x: format_log_to_str([x["intermediate_steps"]])
    # format_log_to_str는 다음 규칙을 사용해서 프롬프트로 유도했던 방식과 동일하게 변환
    # AgentAction.log → Thought
    # AgentAction.tool → Action
    # AgentAction.tool_input → Action Input
    # str (tuple의 두 번째 값, 도구 실행 결과) → Observation

    # RunnablePassThrough.assin()의 결과는
    # { "input": ..., "intermediate_steps": [], 'agent_scratchpad': ...}

) | prompt_template.partial(
    tools=tools_info, tool_names=tool_names
    # 프롬프트 템플릿의 tools, tool_names 변수가 채워지고
    # input, intermediate_steps, agent_scratchpad도 해당 변수가 있으면 채워지게 됨
) | llm | CustomReactOutputParser()


# 어떤 체인 내부에 언어 모델이 있고 그 출력을 받아서 AgentAction/AgentFinish를 리턴하면 그 체인은 Agent
# 따라서 agent_chain은 Agent
# create_react_agent()로 만든 Agent는 ReActSingleInputOutputParser(또는 지정한 커스터마이즈된 OutputParser)가 디폴트 파서

agent_executor = AgentExecutor(
    # agent_chain의 최종 결과물인 AgentAction/AgentFinish는 agent_executor에 자동으로 전달
    # AgentFinish이면 루프를 종료하고 최종 결과를 반환
    # AgentAction이면 거기에 담긴 tool, tool_input을 꺼내서 해당 도구를 실행
    # agent_executor는 내부 intermediate_steps에 전달받은 AgentAction과 도구 실행의 결과를 기록
    
    agent=agent_chain,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    # max_iterations=5,  # 최대 반복 횟수를 5
    # max_execution_time=60 # 최대 실행 시간을 60초로 제한

    # iteration의 정의는
    # 시작은 AgentAction 또는 AgentFinish를 받는 순간
    # 끝은 도구 호출을 하고 intermediate_steps에 기록을 하거나 AgentFinish이어서 답변을 반환하고 루프를 끝내는 순간
)

response = agent_executor.invoke({"input": "5 더하기 3을 한 다음에 그 결과에 2를 곱해줘."})
# agent_executor의 invoke는 input도 넘기지만 내부의 intermediate_steps도 넘김
# { "input": ..., "intermediate_steps": [] }
print(f"\n최종 답변: {response['output']}")

In [16]:


# 수정한 코드
# 수정한 코드
# 수정한 코드
# 수정한 코드
# 수정한 코드

import re
import ast
from langchain.agents.agent import AgentOutputParser
from langchain_core.agents import AgentAction, AgentFinish
from langchain_core.exceptions import OutputParserException
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import AgentExecutor
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool

# AgentOutputParser는 추상 클래스로서 parse() 메서드를 구현해야 함
# parse() 메서드를 구현함으로써 내가 정의한 방식대로 언어 모델의 출력을 해석하는 Agent를 만들 수 있음
# parse는 맘대로 구현해도 되지만 parse의 결과물은 AgentAction/AgentFinish 객체라는 조건은 지켜야 Agent 역할을 할 수 있음

class CustomReactOutputParser(AgentOutputParser):
    # Agent의 역할 구현 및 파싱 방법 커스터마이즈
    def parse(self, text: str) -> AgentAction | AgentFinish:
        # 입력 text는 llm | CustomReactOutputParser()에서 언어 모델의 출력
        
        # 리턴 타입이 AgentAction >>> 다음 단계가 도구 호출을 의미
        # 리턴 타입이 AgentFinish >>> 최종 답변을 찾았고 더 이상의 도구 호출은 필요 없음을 의미
        if "Final Answer:" in text:
            final_answer = text.split("Final Answer:")[-1].strip()
            # text.split("Final Answer:")는
            # 문자열 "Final Answer:"을 기준으로 분리해서 리스트로 만듬
            # "...Final Answer: 16"은 ["...", " 16"]가 됨
            # text.split("Final Answer:")[-1]는 " 16"
            # text.split("Final Answer:")[-1].strip()는 "16"
            return AgentFinish(
                return_values={"output": final_answer},
                log=text,
                # {
                #  "return_values": {"output": "현재 서울의 날씨는 맑고 25도입니다."},
                #  "log": "I now know the final answer after looking up the weather and am ready to respond."
                # }
            )
        action_re = re.compile(r"Action: (.*?)[\n\r]*Action Input: (.*)", re.DOTALL)
        # re.compile(A, B)
        # A는 정규 표현식, B는 정규 표현식의 동작 방식을 지정
        # re.IGNORECASE 또는 re.I: 대소문자를 무시하고 일치
        # re.DOTALL 또는 re.S: 정규 표현식에서 . 은 기본적으로 한 줄 내의 문자를 의미 (H.w는 How와 일치)
        # Hello.*How에 re.DOTALL 플래그를 사용하면 . 패턴에 줄바꿈도 포함하라는 의미
        # re.MULTILINE 또는 re.M: ^와 $가 문자열의 시작과 끝뿐만 아니라 각 줄의 시작과 끝에도 일치

        # "Action: (.*?)[\n\r]*Action Input: (.*)"
        
        # *?는 가능한 한 가장 짧게 일치시키는 것
        # 정규표현식     > 입력 문자열                                  > 결과
        # (Hello.*How) > Hello, world! How are you? How do you do? > Hello, world! How are you? How
        # (Hello.*?How)> Hello, world! How are you? How do you do? > Hello, world! How

        # [\n\r]*는 줄바꿈 문자(\n) 또는 캐리지 리턴 문자(\r)가 0번 이상 반복
        
        # Action: (.*?)[\n\r]*는 Action: ■■■■■ 부분
        # Action Input: (.*)에서 .은 re.DOTALL에 의해서 다음 줄을 포함하여 문자열의 끝까지 모든 내용을 캡처

        match = action_re.search(text)
        # match.group(1)은 Action: 뒤에 오는 문자열
        # match.group(2)은 Action Input: 뒤에 오는 문자열
        if not match:
            raise OutputParserException(f"Could not parse LLM output: `{text}`")
        action = match.group(1).strip()
        action_input_str = match.group(2).strip()

        # 수정된 부분
        # 수정된 부분
        # 수정된 부분
        # 수정된 부분
        # 수정된 부분

        tool_input = {}
        try:
            # 첫 번째 시도: ast.literal_eval로 딕셔너리 문자열을 평가
            evaluated_input = ast.literal_eval(action_input_str)
            if isinstance(evaluated_input, dict):
                tool_input = evaluated_input
            else:
                # ast.literal_eval이 실패하거나 딕셔너리가 아닌 경우
                raise ValueError
        except (SyntaxError, ValueError, TypeError):
            # 두 번째 시도: '키=값' 패턴을 정규 표현식으로 찾아서 딕셔너리로 변환
            key_value_pairs = re.findall(r"(\w+)\s*=\s*([0-9.]+)", action_input_str)
            if not key_value_pairs:
                raise OutputParserException(f"Could not parse action input: `{action_input_str}`")

            try:
                # key_value_pairs가 비어 있지 않으면, 딕셔너리 생성
                tool_input = {key: int(value) for key, value in key_value_pairs}
            except ValueError:
                raise OutputParserException(f"Could not convert action input values to integers: `{action_input_str}`")

        return AgentAction(tool=action, tool_input=tool_input, log=text)

# 아래 셀에서 계속

In [17]:

# 수정한 코드
# 수정한 코드
# 수정한 코드
# 수정한 코드
# 수정한 코드

@tool
def add_numbers(a: int, b: int) -> int:
    """두 개의 정수를 더합니다. 예시: `add_numbers(a=2, b=3)` -> 5"""
    return a + b

@tool
def multiply_numbers(a: int, b: int) -> int:
    """두 개의 정수를 곱합니다. 예시: `multiply_numbers(a=2, b=3)` -> 6"""
    return a * b

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
tools = [add_numbers, multiply_numbers]
tool_names = ", ".join([t.name for t in tools])
tools_info = "\n".join([f"{t.name}: {t.description}" for t in tools])

prompt_template = hub.pull("hwchase17/react")

agent_chain = RunnablePassthrough.assign(
    agent_scratchpad=lambda x: format_log_to_str(x["intermediate_steps"])
) | prompt_template.partial(
    tools=tools_info, tool_names=tool_names
) | llm | CustomReactOutputParser()

agent_executor = AgentExecutor(
    agent=agent_chain,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
)

response = agent_executor.invoke({"input": "5 더하기 3을 한 다음에 그 결과에 2를 곱해줘."})
print(f"\n최종 답변: {response['output']}")



> Entering new AgentExecutor chain...
먼저 5와 3을 더해야 합니다. 그런 다음 그 결과에 2를 곱해야 합니다.
Action: add_numbers
Action Input: a=5, b=3
Observation: 8
Thought: 이제 8에 2를 곱해야 합니다.
Action: multiply_numbers
Action Input: a=8, b=2
Observation: 16
Thought: 이제 최종 답을 알았습니다.
Final Answer: 16


> Finished chain.

최종 답변: 16
